In [1]:
import getpass, os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import TokenTextSplitter
from langchain_community.vectorstores import FAISS

from google import genai
from google.genai import types as genai_types

import re

from pathlib import Path
# Asegúrate de tener instalado pypdf 
from langchain_community.document_loaders import PyPDFLoader


In [2]:
load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")
if not os.environ.get("GOOGLE_EMBEDDING_MODEL"):
    os.environ["GOOGLE_EMBEDDING_MODEL"] = getpass.getpass("Enter Embedding model name: ")
if not os.environ.get("GOOGLE_FAST_MODEL"):
    os.environ["GOOGLE_FAST_MODEL"] = getpass.getpass("Enter Google model name: ")
if not os.environ.get("GOOGLE_LLM_SEED"):
    os.environ["GOOGLE_LLM_SEED"] = getpass.getpass("Enter Google model seed: ")
if not os.environ.get("GOOGLE_LLM_TEMPERATURE"):
    os.environ["GOOGLE_LLM_TEMPERATURE"] = getpass.getpass("Enter Google llm temperature: ")
if not os.environ.get("GOOGLE_LLM_TOPK"):
    os.environ["GOOGLE_LLM_TOPK"] = getpass.getpass("Enter Google llm top k : ")
if not os.environ.get("GOOGLE_LLM_TOPP"):
    os.environ["GOOGLE_LLM_TOPP"] = getpass.getpass("Enter Google llm top p : ")
if not os.environ.get("MAX_OUTPUT_TOKENS_PER_QUESTION"):
    os.environ["MAX_OUTPUT_TOKENS_PER_QUESTION"] = getpass.getpass("Enter Google llm max output tokens per question : ")
if not os.environ.get("GOOGLE_LOGPROBS_ACTIVE"):
    os.environ["GOOGLE_LOGPROBS_ACTIVE"] = getpass.getpass("Enter Google llm log probs (True or False) : ")
if not os.environ.get("CHUNK_SIZE"):
    os.environ["CHUNK_SIZE"] = getpass.getpass("Enter chunk size : ")
if not os.environ.get("CHUNK_OVERLAP"):
    os.environ["CHUNK_OVERLAP"] = getpass.getpass("Enter chunk overlap : ")

API_KEY = os.environ["GOOGLE_API_KEY"]
EMBEDDING_MODEL = os.environ["GOOGLE_EMBEDDING_MODEL"]
LLM_MODEL = os.environ["GOOGLE_FAST_MODEL"]
LLM_SEED = int(os.environ["GOOGLE_LLM_SEED"])
LLM_TEMPERATURE = float(os.environ["GOOGLE_LLM_TEMPERATURE"])
LLM_TOPP = float(os.environ["GOOGLE_LLM_TOPP"])
LLM_TOPK = int(os.environ["GOOGLE_LLM_TOPK"])
OUTPUT_TOKENS_PER_QUESTION = int(os.environ["MAX_OUTPUT_TOKENS_PER_QUESTION"])
ACTIVE_LOGPROBS = True if 'true' == os.environ["GOOGLE_LOGPROBS_ACTIVE"] else False
CHUNK_SIZE = int(os.environ["CHUNK_SIZE"])
CHUNK_OVERLAP = int(os.environ["CHUNK_OVERLAP"])

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Llm model: {LLM_MODEL}")

google_embedding = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)
llm_client = genai.Client()

print("Embeddings y LLM listos.")

Embedding model: gemini-embedding-001
Llm model: gemini-3.1-flash-lite
Embeddings y LLM listos.


In [3]:
pdf_path = Path("../../data/TENERIFE.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {pdf_path.resolve()}")

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()

# Limpiamos mediante regex caracteres extraños asociados a las imagenes
for doc in docs:
    doc.page_content = re.sub(r'\[.*?\]', '', doc.page_content)  # elimina [cualquier cosa]
    doc.page_content = re.sub(r'\(.*?\)', '', doc.page_content)  # elimina (cualquier cosa)
    doc.page_content = re.sub(r'^\s+$', '', doc.page_content, flags=re.MULTILINE)  # líneas solo con espacios
    doc.page_content = re.sub(r'\uf0b7', '', doc.page_content)            # bullet points raros (•)
    doc.page_content = re.sub(r'^Página \d+.*$', '', doc.page_content, flags=re.MULTILINE)  # "Página 1 de 10"
    doc.page_content = re.sub(r'\ufffd', '', doc.page_content)
    doc.page_content = re.sub(r'\n{3,}', '\n\n', doc.page_content)  # múltiples líneas vacías → una
    doc.page_content = re.sub(r' {3,}', ' ', doc.page_content)       # espacios excesivos → uno
    doc.page_content = re.sub(r' {2,}', ' ', doc.page_content)
    doc.page_content = doc.page_content.strip()

docs = [doc for doc in docs if doc.page_content.strip()]

print(f"Documentos cargados: {len(docs)} (una entrada por página).")
print(f"Longitud de la primera página: {len(docs[0].page_content)} caracteres")
print("Muestra del contenido de la primera página:")
for i, page in enumerate(docs):
    print(f"Page {i} - content length: {len(page.page_content)}")


Documentos cargados: 24 (una entrada por página).
Longitud de la primera página: 815 caracteres
Muestra del contenido de la primera página:
Page 0 - content length: 815
Page 1 - content length: 43
Page 2 - content length: 462
Page 3 - content length: 352
Page 4 - content length: 807
Page 5 - content length: 112
Page 6 - content length: 406
Page 7 - content length: 1155
Page 8 - content length: 201
Page 9 - content length: 1205
Page 10 - content length: 1623
Page 11 - content length: 162
Page 12 - content length: 344
Page 13 - content length: 883
Page 14 - content length: 713
Page 15 - content length: 530
Page 16 - content length: 332
Page 17 - content length: 123
Page 18 - content length: 92
Page 19 - content length: 1010
Page 20 - content length: 502
Page 21 - content length: 384
Page 22 - content length: 2178
Page 23 - content length: 471


In [47]:
docs[23].page_content

'OTRAS VAINAS \n\n• Sitios para comer  \no Zona Norte \n▪ Terrazas del Sauzal (zona de El Sauzal  – entre La Laguna y \nLa Orotava. Tienen unas vistas espectaculares y está dentro de la Guía \nMichelín).  \n▪ La Baranda (zona de El Sauzal . Otro que tiene unas vistas \nespectaculares. Este es ideal para desayunar o merendar.)  \n▪ Restaurante Las Vistas (zona de Santa Úrsula  – cerca de La \nOrotava).  \n▪ El Calderito de la Abuela .  \n\n▪ Guachinche El Cubano (zona de Santa Úrsula . Para llegar \natravesaréis una especie de camino de tierra, meteos sin miedo. Si \nvais a Tenerife y no probáis una experiencia en Guachinche, no habéis \nido a Tenerife. Os podría recomendar toda la carta, prácticamente – \nde hecho, no. No podría, ya que en un guachinche no hay carta. Y de \nbebidas solo hay tres cosas: vino, seven up para mezclar con el vino, y \nagua. That’s all. Mi recomendación realmente es que preguntéis a los \ncamareros y, si es vuestra primera experiencia en un guachinche, \nhac

In [19]:
metadatas = [doc.metadata for doc in docs]
content_by_page = [doc.page_content for doc in docs]
words_by_page = [content.split() for content in content_by_page]

In [20]:
max_val = max(len(content) for content in content_by_page)
min_val = min(len(content) for content in content_by_page)
avg_val_page = sum(len(content) for content in content_by_page) / len([len(content) for content in content_by_page])
print(f"Max pÇage content lenght: {max_val}")
print(f"Min page content lenght: {min_val}")
print(f"Avg page content lenght: {avg_val_page}")

Max pÇage content lenght: 2178
Min page content lenght: 43
Avg page content lenght: 621.0416666666666


In [4]:
# SPLITTER 1 -> Spliter by text
# chunk: 590
# overlap: 245
splitter_text = RecursiveCharacterTextSplitter(
    chunk_size=850,
    chunk_overlap=400,
    add_start_index=True
)

splitter_text_list = splitter_text.split_documents(docs)
for i, doc in enumerate(splitter_text_list):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = pdf_path.name

print(f"{splitter_text_list}")
print(f"Number of splitters: {len(splitter_text_list)}")

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'title': 'Microsoft Word - TENERIFE.docx', 'moddate': '2025-07-13T20:00:01+00:00', 'source': '..\\..\\data\\TENERIFE.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'start_index': 0, 'chunk_id': 0, 'source_name': 'TENERIFE.pdf'}, page_content='TENERIFE – LUGARES DE INTERÉS \nSITIOS QUE VER \n\nZONA NORTE \n\n• Santa Cruz de Tenerife: \nSanta Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa \nCruz sería: \n- Aparcar en el aparcamiento del Parque Marítimo . \n- Caminar por la Avenida Marítima hasta Plaza de España . \n- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife . \n- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo \ndirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - \nubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del \nPríncipe - ub

In [11]:
# SPLITTER 2 -> Splitter by character
splitter_char = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=13,
    add_start_index=True,
    strip_whitespace=True
)

splitter_char_list = splitter_char.split_documents(docs)
print(f"{splitter_char_list}")
print(f"Number of splitters: {len(splitter_char_list)}")


[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'title': 'Microsoft Word - TENERIFE.docx', 'moddate': '2025-07-13T20:00:01+00:00', 'source': '../../data/TENERIFE.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='TENERIFE – LUGARES DE INTERÉS \nSITIOS QUE VER \n \nZONA NORTE \n \n• Santa Cruz de Tenerife: \nSanta Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa \nCruz sería: \n- Aparcar en el aparcamiento del Parque Marítimo (ubicación). \n- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). \n- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). \n- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo \ndirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - \nubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del \nPríncipe - ubicación). \n- 

In [ ]:
# SPLITTER 3 -> Splitter by token
# get tokens in PDF
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode("".join(content_by_page))
print(len(tokens))  # → número exacto de tokens
print(tokens)       # → [1027, 6926, 20541, 38, 11MB, ...]
tokens_by_page = [len(enc.encode(page)) for page in content_by_page]
total_tokens = sum(tokens_by_page)
avg_token_by_page = total_tokens / len(content_by_page)
print(f"Avg token by page: {avg_token_by_page}")

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=100,
    chunk_overlap=94,
    strip_whitespace=True
)

splitter_token_list = token_splitter.split_documents(docs)

print(f"{splitter_token_list}")
print(f"Number of splits: {len(splitter_token_list)}")

4810
[51, 12545, 40777, 1389, 445, 3014, 4577, 50, 3467, 30442, 27887, 50, 720, 50, 964, 29236, 30776, 30361, 33006, 57, 96087, 70188, 2505, 33006, 6806, 16376, 21510, 409, 350, 804, 1643, 25, 720, 64248, 21510, 409, 350, 804, 1643, 1560, 1208, 6864, 409, 1208, 374, 4355, 13, 42248, 7206, 1208, 68563, 264, 58548, 4502, 91507, 264, 16376, 720, 34, 63423, 1446, 7583, 25, 720, 12, 362, 1768, 7063, 665, 658, 1469, 8362, 23666, 1624, 4366, 593, 2947, 16485, 11620, 320, 42281, 5840, 570, 720, 12, 8215, 14080, 4247, 1208, 362, 88295, 2947, 16485, 7675, 29626, 41407, 409, 57608, 320, 42281, 5840, 570, 720, 12, 20388, 658, 84544, 409, 1208, 362, 88295, 2947, 16485, 7675, 11, 2807, 658, 54406, 822, 409, 350, 804, 1643, 320, 42281, 5840, 570, 720, 12, 44071, 21205, 33602, 5670, 264, 41407, 57608, 11, 68935, 73, 686, 653, 39894, 320, 2008, 404, 1208, 3400, 273, 96783, 720, 64684, 22776, 41407, 468, 1216, 1565, 482, 63874, 5840, 1389, 26, 6348, 58009, 658, 4366, 593, 85341, 5960, 370, 4298, 482, 72

In [5]:
# Generate embeddings and save it as vectos with FAISS
vector_store = FAISS.from_documents(
    documents=splitter_text_list,
    embedding=google_embedding
)

In [27]:
# TESTING EMBEDDINGS
def mostrar_documentos_recuperados(documentos):
    for i, doc in enumerate(documentos, start=1):
        doc_content = doc[0]
        score = doc[1]
        print(f"Resultado {i}")
        print("Fuente:", doc_content.metadata.get("source_name"))
        print("Página:", doc_content.metadata.get("page"))
        print("Chunk:", doc_content.metadata.get("chunk_id"))
        print("Similarity: ", score)
        print(doc_content.page_content)
        print("-" * 80)

pregunta_experimento = "¿Dónde puedo ir de Discoteca?"

for k in [2, 3, 4, 6]:
    print(f"\n\nK = {k}")
    print("=" * 80)
    docs_k = vector_store.similarity_search_with_relevance_scores(pregunta_experimento, k=k)
    mostrar_documentos_recuperados(docs_k)



K = 2
Resultado 1
Fuente: TENERIFE.pdf
Página: 18
Chunk: 23
Similarity:  0.5705042
También abre durante el día como restaurante y cocktail bar por si os queréis 
tomar algo aquí. Durante el día es algo así:
--------------------------------------------------------------------------------
Resultado 2
Fuente: TENERIFE.pdf
Página: 17
Chunk: 22
Similarity:  0.5591124
También se encuentra al lado de esta playa el Papagayo Beach Club, el cual 
ha sido elegido varios años como mejor beach club de España y al que tenéis 
que ir sí o sí si queréis salir de fiesta por Tenerife . Podéis consultar las 
fiestas del Papagayo aquí , 
también consultad sus redes sociales y esas vainas para estar al tanto.
--------------------------------------------------------------------------------


K = 3
Resultado 1
Fuente: TENERIFE.pdf
Página: 18
Chunk: 23
Similarity:  0.5705042
También abre durante el día como restaurante y cocktail bar por si os queréis 
tomar algo aquí. Durante el día es algo así:
----------

In [6]:
# Configuraciones el modelo sin tools
with open("../../data/templates/system_instructions.txt", "r", encoding="utf-8") as f:
    instruction_content = f.read()
    model_config = genai_types.GenerateContentConfig(
            system_instruction=instruction_content,
            response_modalities=["TEXT"],
            temperature=LLM_TEMPERATURE,
            top_k=LLM_TOPK,
            top_p=LLM_TOPP,
            seed=LLM_SEED,
            max_output_tokens=OUTPUT_TOKENS_PER_QUESTION,
            response_logprobs=ACTIVE_LOGPROBS,
            thinking_config=genai_types.GenerationConfigThinkingConfig(
               thinking_level=genai_types.ThinkingLevel.MEDIUM
            )
        )


In [ ]:
# TESTING LLM WITH RAG WITHOUT TOOLS

def format_docs_with_sources(documentos):
    bloques = []
    for i, doc in enumerate(documentos, start=1):
        source = doc.metadata.get("source_name", "fuente desconocida")
        page = doc.metadata.get("page", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        bloques.append(
            f"""[Fuente {i}: {source}, página {page}, chunk {chunk_id}]
                {doc.page_content}
            """
        )
    return "".join(bloques)

def answer_with_rag(question: str, k: int = 3, show_context: bool = False):
    retrieved_docs = vector_store.similarity_search(question, k=k)
    context = format_docs_with_sources(retrieved_docs)

    if show_context:
        print("CONTEXTO RECUPERADO")
        print(context)
        print("=" * 80)

    response = llm_client.models.generate_content(
        model=LLM_MODEL,
        contents=f"context: {context}; user question: {question}",
        config=model_config
    )

    return {
        "question": question,
        "answer": response.text,
        "context": retrieved_docs,
        "prompt": f"context: {context}; user question: {question}"
    }

def average_output_tokens(results):
    """Calcula el promedio de tokens de salida consumidos por una lista de resultados de answer_with_rag.

    Util para estimar un valor razonable de max_output_tokens en base al
    consumo real observado en un conjunto de preguntas de prueba.
    """
    token_counts = [r["output_tokens"] for r in results if r.get("output_tokens") is not None]

    if not token_counts:
        return {"avg": 0, "min": 0, "max": 0, "count": 0}

    return {
        "avg": sum(token_counts) / len(token_counts),
        "min": min(token_counts),
        "max": max(token_counts),
        "count": len(token_counts),
    }

test_questions = [
    "¿Cómo puedo subir al Teide?",
    "Dame un listado de playas",
    "¿Qué discotecas hay en Tenerife?",
    "¿Qué puedo comer?",
    "Qué coches hay disponibles para alquilar"
]

results = []
for question in test_questions:
    print("PREGUNTA:", question)
    result = answer_with_rag(question, k=3)
    results.append(result)
    print(result["answer"])
    print(f"Output tokens: {result['output_tokens']}")
    print("-" * 100)

output_tokens_stats = average_output_tokens(results)
print("ESTADISTICAS DE TOKENS DE SALIDA")
print(f"Promedio: {output_tokens_stats['avg']:.2f}")
print(f"Minimo:   {output_tokens_stats['min']}")
print(f"Maximo:   {output_tokens_stats['max']}")
print(f"Muestras: {output_tokens_stats['count']}")

PREGUNTA: ¿Cómo puedo subir al Teide?
Para subir al Teide, se recomienda tomar la carretera TF24 desde la rotonda de Padre Anchieta en La Laguna, haciendo paradas en el Mirador de La Tarta o el de Chipeque. Una vez en la zona, lo ideal es aparcar en el Parador de las Cañadas del Teide para visitar el mirador de La Ruleta. Para llegar hasta el pico, puedes utilizar los teleféricos, y para obtener más información, puedes visitar el Centro de Visitantes de El Portillo.

Fuente: TENERIFE.pdf, páginas 14-15.
Output tokens: 126
----------------------------------------------------------------------------------------------------
PREGUNTA: Dame un listado de playas
Aquí tienes una lista de playas en Tenerife mencionadas en la guía: Playa de Los Patos, Playa del Ancón, El Médano, La Tejita, Playa de Los Cristianos, Playa de Las Vistas y Playa de Benijo.

Fuente 1, 2, 3
Output tokens: 60
----------------------------------------------------------------------------------------------------
PREGUNTA:

In [9]:
# VER EL LIMITE DE TOKENS DE INPUT Y OUTPUT DE UN MODELO DE GOOGLE
print("\nGEMINI 3.1 FLASH LITE:")
model_info = llm_client.models.get(model=LLM_MODEL)
print(f"{model_info.input_token_limit = :,.0f}")
print(f"{model_info.output_token_limit = :,.0f}")
max_output_tokens_model = model_info.output_token_limit;


GEMINI 3.1 FLASH LITE:
model_info.input_token_limit = 1,048,576
model_info.output_token_limit = 65,536


In [8]:
def calcular_tokens_gemini(
    client,
    model_name: str,
    contents,
    response=None
) -> dict:
    """
    Calcula el consumo de tokens usando el tokenizador nativo de Gemini.
    
    Args:
        client     : instancia de genai.Client
        model_name : nombre del modelo usado
        contents   : mensaje enviado (string o lista de parts)
        response   : respuesta del modelo (opcional)
    
    Returns:
        dict con input_tokens, output_tokens, total_tokens y fuente
    """

    # ── Input tokens — siempre exacto con count_tokens ────────
    count_response = client.models.count_tokens(
        model=model_name,
        contents=contents
    )
    input_tokens = count_response.total_tokens

    # ── Output tokens ──────────────────────────────────────────
    output_tokens = 0

    if response is not None:
        # Caso 1: usage_metadata disponible → exacto
        if response.usage_metadata is not None:
            output_tokens = response.usage_metadata.candidates_token_count or 0
            fuente = "gemini_native"

        # Caso 2: usage_metadata None → contar tokens del texto de respuesta
        else:
            if response.candidates:
                for candidate in response.candidates:
                    if candidate.content and candidate.content.parts:
                        for part in candidate.content.parts:
                            if hasattr(part, "text") and part.text:
                                # count_tokens sobre el texto de la respuesta
                                count_output = client.models.count_tokens(
                                    model=model_name,
                                    contents=part.text
                                )
                                output_tokens += count_output.total_tokens
            fuente = "gemini_count_tokens"
    else:
        fuente = "gemini_count_tokens"

    return {
        "input_tokens"  : input_tokens,
        "output_tokens" : output_tokens,
        "total_tokens"  : input_tokens + output_tokens,
        "fuente"        : fuente
    }

In [13]:
# Following conversation with max output tokens control

chat = llm_client.chats.create(
    model=LLM_MODEL,
    config=model_config,
)

output_tokens_remain = max_output_tokens_model
first_message = "Una vez vistado el Teide, ¿Donde se puede comer?"
followup_message = "Ahora, quiero ir a bañarme, ¿Qué playas me recomiendas?"

llm_first_response = chat.send_message(message = first_message)

consumed_tokens_information = calcular_tokens_gemini(llm_client, LLM_MODEL, first_message, llm_first_response)
output_tokens_remain -= consumed_tokens_information['output_tokens']
print(f"Input consumido : {consumed_tokens_information['input_tokens']}")
print(f"Output consumido : {consumed_tokens_information['output_tokens']}")
print(f"Output restante  : {output_tokens_remain}")

llm_followsecond_response = chat.send_message(
    config=genai_types.GenerateContentConfig(
        max_output_tokens=output_tokens_remain
    ),
    message = followup_message)

consumed_tokens_information = calcular_tokens_gemini(llm_client, LLM_MODEL, first_message, llm_followsecond_response)
output_tokens_remain -= consumed_tokens_information['output_tokens']
print(f"Input consumido : {consumed_tokens_information['input_tokens']}")
print(f"Output consumido : {consumed_tokens_information['output_tokens']}")
print(f"Output restante  : {output_tokens_remain}")

print("CONVERSATION:")
for message in chat.get_history():
    print(f"\n[{message.role.title()}]", end=": ")
    print(message.parts[0].text)

Input consumido : 16
Output consumido : 22
Output restante  : 65514
Input consumido : 16
Output consumido : 719
Output restante  : 64795
CONVERSATION:

[User]: Una vez vistado el Teide, ¿Donde se puede comer?

[Model]: Lo siento, pero no dispongo de información sobre lugares para comer cerca del Teide en los documentos proporcionados.

[User]: Ahora, quiero ir a bañarme, ¿Qué playas me recomiendas?

[Model]: Tenerife ofrece playas para todos los gustos, dependiendo de la zona donde te encuentres al bajar del Teide. Aquí te clasifico algunas de las mejores opciones según su estilo:

### 1. Si bajas hacia el SUR (Zona más turística y soleada)
Al bajar del Teide por el sur (vía Vilaflor o Chío), llegarás a zonas con playas de arena dorada y aguas más tranquilas:
*   **Playa del Duque (Costa Adeje):** Es una de las más elegantes, con arena clara, aguas cristalinas y todos los servicios cercanos (hamacas, restaurantes, paseos).
*   **Playa de Las Vistas (Los Cristianos):** Es muy amplia, co

In [9]:
# TOOL
import unicodedata

# --- Simulated Tenerife mobility dataset (real towns from TENERIFE.pdf, invented but plausible data) ---

VALID_TRANSPORT_MODES = ("bus", "car", "cycle", "walk")

# Ciudad/POI -> nombre legible + pueblo más cercano donde se coge el transporte
CITIES = {
    "santa_cruz":         {"display": "Santa Cruz de Tenerife", "nearest_town": "Santa Cruz de Tenerife"},
    "la_laguna":          {"display": "La Laguna",              "nearest_town": "La Laguna"},
    "puerto_de_la_cruz":  {"display": "Puerto de la Cruz",      "nearest_town": "Puerto de la Cruz"},
    "la_orotava":         {"display": "La Orotava",             "nearest_town": "La Orotava"},
    "santa_ursula":       {"display": "Santa Úrsula",           "nearest_town": "Santa Úrsula"},
    "icod":               {"display": "Icod de los Vinos",      "nearest_town": "Icod de los Vinos"},
    "garachico":          {"display": "Garachico",              "nearest_town": "Garachico"},
    "buenavista":         {"display": "Buenavista del Norte",   "nearest_town": "Buenavista del Norte"},
    "masca":              {"display": "Masca",                  "nearest_town": "Buenavista del Norte"},
    "los_gigantes":       {"display": "Los Gigantes",           "nearest_town": "Santiago del Teide"},
    "adeje":              {"display": "Adeje",                  "nearest_town": "Adeje"},
    "los_cristianos":     {"display": "Los Cristianos",         "nearest_town": "Arona"},
    "el_teide":           {"display": "El Teide",               "nearest_town": "La Orotava"},
    "punta_de_teno":      {"display": "Punta de Teno",          "nearest_town": "Buenavista del Norte"},
    "anaga":              {"display": "Anaga",                  "nearest_town": "La Laguna"},
}

# Alias frecuentes -> clave canónica (formas cortas/largas que un usuario o LLM puede usar)
CITY_ALIASES = {
    "teide": "el_teide",
    "el teide": "el_teide",
    "laguna": "la_laguna",
    "san cristobal de la laguna": "la_laguna",
    "puerto": "puerto_de_la_cruz",
    "puerto cruz": "puerto_de_la_cruz",
    "orotava": "la_orotava",
    "santa ursula": "santa_ursula",
    "icod": "icod",
    "icod de los vinos": "icod",
    "buenavista": "buenavista",
    "buenavista del norte": "buenavista",
    "cristianos": "los_cristianos",
    "gigantes": "los_gigantes",
    "santa cruz": "santa_cruz",
    "santa cruz de tenerife": "santa_cruz",
    "punta de teno": "punta_de_teno",
    "teno": "punta_de_teno",
}

# Tiempo estimado en minutos por modo para cada par (no dirigido). None = modo no viable.
TRAVEL_TIMES = {
    frozenset({"puerto_de_la_cruz", "santa_cruz"}): {"bus": 55, "car": 35, "cycle": 150, "walk": None},
    frozenset({"puerto_de_la_cruz", "la_orotava"}): {"bus": 20, "car": 12, "cycle": 35,  "walk": 75},
    frozenset({"puerto_de_la_cruz", "la_laguna"}):  {"bus": 50, "car": 30, "cycle": 140, "walk": None},
    frozenset({"la_laguna", "santa_cruz"}):         {"bus": 25, "car": 15, "cycle": 45,  "walk": 110},
    frozenset({"la_orotava", "santa_ursula"}):      {"bus": 18, "car": 10, "cycle": 30,  "walk": 70},
    frozenset({"la_orotava", "el_teide"}):          {"bus": None, "car": 55, "cycle": 180, "walk": None},
    frozenset({"icod", "garachico"}):               {"bus": 15, "car": 9,  "cycle": 25,  "walk": 60},
    frozenset({"garachico", "buenavista"}):         {"bus": 25, "car": 16, "cycle": 40,  "walk": None},
    frozenset({"buenavista", "masca"}):             {"bus": 35, "car": 25, "cycle": 80,  "walk": None},
    frozenset({"buenavista", "punta_de_teno"}):     {"bus": 30, "car": 20, "cycle": 45,  "walk": None},
    frozenset({"adeje", "los_cristianos"}):         {"bus": 20, "car": 12, "cycle": 35,  "walk": 80},
    frozenset({"adeje", "los_gigantes"}):           {"bus": 40, "car": 25, "cycle": 70,  "walk": None},
    frozenset({"santa_cruz", "adeje"}):             {"bus": 75, "car": 50, "cycle": None, "walk": None},
    frozenset({"la_laguna", "anaga"}):              {"bus": 45, "car": 30, "cycle": 90,  "walk": None},
}

# Línea de guagua (TITSA) y punto de recogida por par no dirigido (solo donde el bus es viable).
BUS_LINES = {
    frozenset({"puerto_de_la_cruz", "santa_cruz"}): {"line": "TITSA 103", "stop": "Estación de guaguas de Puerto de la Cruz"},
    frozenset({"puerto_de_la_cruz", "la_orotava"}): {"line": "TITSA 350", "stop": "Estación de guaguas de Puerto de la Cruz"},
    frozenset({"puerto_de_la_cruz", "la_laguna"}):  {"line": "TITSA 102", "stop": "Estación de guaguas de Puerto de la Cruz"},
    frozenset({"la_laguna", "santa_cruz"}):         {"line": "TITSA 015", "stop": "Intercambiador de La Laguna"},
    frozenset({"la_orotava", "santa_ursula"}):      {"line": "TITSA 345", "stop": "Parada Plaza del Ayuntamiento, La Orotava"},
    frozenset({"icod", "garachico"}):               {"line": "TITSA 363", "stop": "Estación de guaguas de Icod de los Vinos"},
    frozenset({"garachico", "buenavista"}):         {"line": "TITSA 363", "stop": "Parada principal de Garachico"},
    frozenset({"buenavista", "masca"}):             {"line": "TITSA 355", "stop": "Estación de Buenavista del Norte"},
    frozenset({"buenavista", "punta_de_teno"}):     {"line": "TITSA 369", "stop": "Estación de Buenavista del Norte"},
    frozenset({"adeje", "los_cristianos"}):         {"line": "TITSA 467", "stop": "Estación de guaguas de Adeje"},
    frozenset({"adeje", "los_gigantes"}):           {"line": "TITSA 473", "stop": "Estación de guaguas de Adeje"},
    frozenset({"santa_cruz", "adeje"}):             {"line": "TITSA 110", "stop": "Intercambiador de Santa Cruz"},
    frozenset({"la_laguna", "anaga"}):              {"line": "TITSA 077", "stop": "Intercambiador de La Laguna"},
}


def _normalize_city(name):
    """Normaliza un nombre de ciudad a su clave canónica (sin acentos, minúsculas, alias)."""
    text = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    if text in CITY_ALIASES:
        return CITY_ALIASES[text]
    key = text.replace(" ", "_")
    return key


def _error(error_code, message, **extra):
    """Construye una respuesta de error homogénea (siempre serializable a JSON)."""
    payload = {"status": "ERROR", "error_code": error_code, "message": message}
    payload.update(extra)
    return payload


def get_city_mobility(origin, destination, transport_mode="best"):
    """Simula una API de movilidad de Tenerife.

    Dado un `origin` y un `destination` (pueblos/POIs de Tenerife) devuelve el mejor
    transporte para llegar, el tiempo estimado en minutos y la línea de guagua / punto de
    recogida en el pueblo más cercano al origen.

    Args:
        origin: Pueblo de origen en Tenerife (p.ej. "Puerto de la Cruz").
        destination: Pueblo o punto de interés de destino (p.ej. "Santa Cruz", "El Teide").
        transport_mode: Modo preferido ("bus", "car", "cycle", "walk") o "best" para
            comparar todos y recomendar el más rápido. Por defecto "best".

    Returns:
        dict serializable a JSON. En éxito: status="OK" con recommended_transport,
        estimated_minutes, nearest_town, bus_line, pickup_point y all_options. En error:
        status="ERROR" con error_code en {BAD_REQUEST, CITY_NOT_FOUND, INTERNAL_SERVER_ERROR}.
    """
    try:
        # --- BAD_REQUEST ---
        if not isinstance(origin, str) or not origin.strip():
            return _error("BAD_REQUEST", "El parámetro 'origin' es obligatorio y no puede estar vacío.")
        if not isinstance(destination, str) or not destination.strip():
            return _error("BAD_REQUEST", "El parámetro 'destination' es obligatorio y no puede estar vacío.")

        mode = (transport_mode or "best").strip().lower()
        if mode not in VALID_TRANSPORT_MODES + ("best",):
            return _error(
                "BAD_REQUEST",
                f"transport_mode '{transport_mode}' no válido. Usa uno de "
                f"{list(VALID_TRANSPORT_MODES) + ['best']}.",
                valid_modes=list(VALID_TRANSPORT_MODES) + ["best"],
            )

        origin_key = _normalize_city(origin)
        dest_key = _normalize_city(destination)

        if origin_key == dest_key:
            return _error("BAD_REQUEST", "El origen y el destino no pueden ser el mismo lugar.")

        # --- CITY_NOT_FOUND ---
        valid_display = [c["display"] for c in CITIES.values()]
        if origin_key not in CITIES:
            return _error("CITY_NOT_FOUND", f"Origen '{origin}' no encontrado en Tenerife.",
                          valid_cities=valid_display)
        if dest_key not in CITIES:
            return _error("CITY_NOT_FOUND", f"Destino '{destination}' no encontrado en Tenerife.",
                          valid_cities=valid_display)

        # --- INTERNAL_SERVER_ERROR: par sin datos de ruta (fallo simulado del backend) ---
        pair = frozenset({origin_key, dest_key})
        if pair not in TRAVEL_TIMES:
            return _error(
                "INTERNAL_SERVER_ERROR",
                f"No se pudo calcular la ruta entre '{CITIES[origin_key]['display']}' y "
                f"'{CITIES[dest_key]['display']}'. Inténtalo de nuevo más tarde.",
            )

        times = TRAVEL_TIMES[pair]
        viable = {m: t for m, t in times.items() if t is not None}
        if not viable:
            return _error(
                "INTERNAL_SERVER_ERROR",
                "No hay ningún modo de transporte disponible para esta ruta.",
            )

        # --- Selección de transporte ---
        best_mode = min(viable, key=viable.get)
        note = None
        if mode == "best":
            chosen = best_mode
        elif mode in viable:
            chosen = mode
        else:
            # Modo solicitado no viable: caemos al mejor disponible y avisamos.
            chosen = best_mode
            note = (f"El modo '{mode}' no está disponible para esta ruta; "
                    f"se recomienda '{best_mode}' en su lugar.")

        bus_info = BUS_LINES.get(pair, {})
        result = {
            "status": "OK",
            "origin": CITIES[origin_key]["display"],
            "destination": CITIES[dest_key]["display"],
            "recommended_transport": chosen,
            "estimated_minutes": viable[chosen],
            "nearest_town": CITIES[origin_key]["nearest_town"],
            "bus_line": bus_info.get("line"),
            "pickup_point": bus_info.get("stop"),
            "all_options": viable,
        }
        if note:
            result["note"] = note
        return result

    except Exception as exc:  # noqa: BLE001 - frontera de la API simulada: nunca propagar
        return _error("INTERNAL_SERVER_ERROR", f"Error inesperado en el servicio de movilidad: {exc}")


print("get_city_mobility lista. Ciudades disponibles:")
print(", ".join(c["display"] for c in CITIES.values()))

get_city_mobility lista. Ciudades disponibles:
Santa Cruz de Tenerife, La Laguna, Puerto de la Cruz, La Orotava, Santa Úrsula, Icod de los Vinos, Garachico, Buenavista del Norte, Masca, Los Gigantes, Adeje, Los Cristianos, El Teide, Punta de Teno, Anaga


In [10]:
# TEST TOOL
import json

demo_calls = [
    ("Ruta normal (modo concreto)", ("Puerto de la Cruz", "Santa Cruz de Tenerife", "bus")),
    ("Comparación 'best'",          ("Adeje", "Los Cristianos")),
    ("Modo no viable -> fallback",  ("La Orotava", "El Teide", "bus")),
    ("CITY_NOT_FOUND",              ("Madrid", "Adeje")),
    ("BAD_REQUEST (mismo lugar)",   ("Adeje", "Adeje")),
    ("BAD_REQUEST (modo inválido)", ("Adeje", "Los Cristianos", "plane")),
]

for title, args in demo_calls:
    result = get_city_mobility(*args)
    print(f"### {title}: get_city_mobility{args}")
    print(json.dumps(result, ensure_ascii=False, indent=2))  # prueba que es serializable a JSON
    print("-" * 70)

### Ruta normal (modo concreto): get_city_mobility('Puerto de la Cruz', 'Santa Cruz de Tenerife', 'bus')
{
  "status": "OK",
  "origin": "Puerto de la Cruz",
  "destination": "Santa Cruz de Tenerife",
  "recommended_transport": "bus",
  "estimated_minutes": 55,
  "nearest_town": "Puerto de la Cruz",
  "bus_line": "TITSA 103",
  "pickup_point": "Estación de guaguas de Puerto de la Cruz",
  "all_options": {
    "bus": 55,
    "car": 35,
    "cycle": 150
  }
}
----------------------------------------------------------------------
### Comparación 'best': get_city_mobility('Adeje', 'Los Cristianos')
{
  "status": "OK",
  "origin": "Adeje",
  "destination": "Los Cristianos",
  "recommended_transport": "car",
  "estimated_minutes": 12,
  "nearest_town": "Adeje",
  "bus_line": "TITSA 467",
  "pickup_point": "Estación de guaguas de Adeje",
  "all_options": {
    "bus": 20,
    "car": 12,
    "cycle": 35,
    "walk": 80
  }
}
----------------------------------------------------------------------

In [11]:
# LLM TOOL SCHEMA
mobility_tool_schema = {
    "type": "function",
    "name": "get_city_mobility",
    "description": "Given a origin, destination and, optionally, a transport mode between bus, car, cycle, walk or best, which is chosen by default. Then, it returns the best transportation mode with best ETA and nearest town from origin",
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "origin": {
                "type": "string",
                "description": "Name of town or city to start visiting destination place. The town or city must be located in Tenerife",
            },
            "destination": {
                "type": "string",
                "description": "Name of place to arrive. The town or city must be located in Tenerife"
            }, 
            "transport_mode": {
                "type": "string",
                "description": "Mode of transport to go to destination. If transport_mode is not indicated, 'best' will be chosen compare all kind of transportation modes and choose the fastest one",
                "default": "best"
            }
        },
        "required": ["origin", "destination"]
    }
} 

def parse_schema(parameters: dict) -> genai_types.Schema:
    
    type_map = {
        "string":  genai_types.Type.STRING,
        "object":  genai_types.Type.OBJECT
    }
    
    schema_type = type_map[parameters["type"]]
    is_object   = schema_type == genai_types.Type.OBJECT

    # Only parse properties for object types
    properties = {}
    if is_object and "properties" in parameters:
        for key, value in parameters["properties"].items():
            properties[key] = parse_schema(value)

    return genai_types.Schema(
        type=schema_type,
        description=parameters.get("description", ""),
        properties=properties   if is_object else None,  # ✅ only for objects
        required=parameters.get("required", []) if is_object else None,  # ✅ only for objects
        enum=None
    )

search_mobility_parsed = genai_types.FunctionDeclaration(
    name=mobility_tool_schema.get("name"),
    description=mobility_tool_schema["description"],
    parameters=parse_schema(mobility_tool_schema["parameters"])
)

search_mobility_tool = genai_types.Tool(
    function_declarations=[search_mobility_parsed]
)

In [12]:
# Configuraciones el modelo con tool
with open("../../data/templates/system_instructions.txt", "r", encoding="utf-8") as f:
    instruction_content = f.read()
    model_config = genai_types.GenerateContentConfig(
            system_instruction=instruction_content,
            response_modalities=["TEXT"],
            temperature=LLM_TEMPERATURE,
            top_k=LLM_TOPK,
            top_p=LLM_TOPP,
            seed=LLM_SEED,
            max_output_tokens=OUTPUT_TOKENS_PER_QUESTION,
            response_logprobs=ACTIVE_LOGPROBS,
            thinking_config=genai_types.GenerationConfigThinkingConfig(
               thinking_level=genai_types.ThinkingLevel.MEDIUM
            ),
            tools=[search_mobility_tool],
            tool_config=genai_types.ToolConfig(
               function_calling_config=genai_types.FunctionCallingConfig(
                   mode=genai_types.FunctionCallingConfigMode.AUTO
                )
                
            )
        )

In [14]:
# TEST LLM WITH TOOL AND TOOL USAGE METRICS

def format_docs_with_sources(documentos):
    bloques = []
    for i, doc in enumerate(documentos, start=1):
        source = doc.metadata.get("source_name", "fuente desconocida")
        page = doc.metadata.get("page", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        bloques.append(
            f"""[Fuente {i}: {source}, página {page}, chunk {chunk_id}]
                {doc.page_content}
            """
        )
    return "".join(bloques)

def tool_usage(llm_response):
    call = None
    tool_result = None
    if llm_response.function_calls is None:
        print("El modelo no ha solicitado ninguna herramienta.")
    
    else:
        function_calls = [item for item in llm_response.function_calls]
        
        if not function_calls:
            print("El modelo no ha solicitado ninguna herramienta.")
        else:
            call = function_calls[0]
            args = dict(call.args)
            print("Herramienta solicitada:", call.name)
            print("Argumentos:", args)

            if call.name == "get_city_mobility":
                tool_result = get_city_mobility(**args)
            else:
                raise ValueError(f"Herramienta no soportada: {call.name}")

            print("Resultado de la herramienta:")
            print(json.dumps(tool_result, indent=2, ensure_ascii=False))
    
    name = call.name if call is not None else None
    tool_response = tool_result if tool_result is not None else None
    
    return (name, tool_response)

def answer_with_rag(chat, question: str, k: int = 3, show_context: bool = False):
    retrieved_docs = vector_store.similarity_search(question, k=k)
    context = format_docs_with_sources(retrieved_docs)

    if show_context:
        print("CONTEXTO RECUPERADO")
        print(context)
        print("=" * 80)

    response = chat.send_message(
        message = f"context: {context}; user question: {question}",
    )

    tool_name, tool_response = tool_usage(response)

    final_response = response
    if tool_response is not None and tool_name is not None:
        final_response = chat.send_message(
        genai_types.Part(
            function_response=genai_types.FunctionResponse(
                name=tool_name,
                response={"result": tool_response}
            )
        )
    )
    return {
        "question": question,
        "answer": final_response.text,
        "context": retrieved_docs,
        "prompt": f"context: {context}; user question: {question}",
        "output_tokens": final_response.usage_metadata.candidates_token_count
    }

test_questions = [
    "¿Cómo puedo subir al Teide?",
    "Dame un listado de playas",
    "¿Qué discotecas hay en Tenerife?",
    "¿Qué puedo comer?",
    "¿Cómo puedo ir desde Madrid hasta Tenerife?",
    "¿Cómo puedo llegar desde la Orotava hasta el Teide?",
    "¿Cómo puedo llegar desde la Orotava hasta el Teide en bicicleta?"
]

results = []
chat = llm_client.chats.create(
    model=LLM_MODEL,
    config=model_config,
)
for question in test_questions:
    print("PREGUNTA:", question)
    result = answer_with_rag(chat, question, k=3)
    results.append(result)
    print(result["answer"])
    print(f"Output tokens: {result['output_tokens']}")
    print("-" * 100)

PREGUNTA: ¿Cómo puedo subir al Teide?
El modelo no ha solicitado ninguna herramienta.
Para subir al Teide, se recomienda tomar la carretera TF-24 desde la rotonda de Padre Anchieta en La Laguna, haciendo paradas en el Mirador de La Tarta y el Mirador de Chipeque. Una vez en la zona, puedes aparcar en el Parador de las Cañadas del Teide para visitar el mirador de La Ruleta. Para acceder al pico, puedes utilizar el teleférico, y si buscas información, el Centro de Visitantes de El Portillo es gratuito. Además, subir de noche cuando el cielo está despejado permite observar uno de los cielos estrellados más espectaculares del mundo.

TENERIFE.pdf, páginas 14-15
Output tokens: 150
----------------------------------------------------------------------------------------------------
PREGUNTA: Dame un listado de playas
El modelo no ha solicitado ninguna herramienta.
Aquí tienes un listado de playas mencionadas en la guía:

*   **Zona Norte:** Playa de Los Patos, Playa del Ancón y Playa de Benij

## EVALUACIÓN: CONJUNTO REPRODUCIBLE DE PREGUNTAS

Conjunto fijo de preguntas para evaluar el pipeline RAG + tool-calling de forma reproducible.
Cubre tres categorías:

- **RAG normal**: preguntas con respuesta esperada en el PDF (TENERIFE.pdf).
- **Fuera de alcance**: preguntas que el PDF no responde, para detectar alucinaciones o verificar que el modelo admite no saber.
- **Tool-calling**: preguntas de movilidad entre lugares de Tenerife que deberían disparar `get_city_mobility`, incluyendo casos válidos e inválidos (ciudad no encontrada).

Ejecutar este conjunto tras cambios de prompt, modelo, `chunk_size`/`chunk_overlap` o `k` permite comparar resultados de forma consistente.

In [15]:
# CONJUNTO REPRODUCIBLE DE PREGUNTAS DE EVALUACIÓN

evaluation_questions = [
    # --- RAG normal: respuesta esperada en TENERIFE.pdf ---
    {"category": "rag_normal", "question": "¿Cómo puedo subir al Teide?"},
    {"category": "rag_normal", "question": "Dame un listado de playas"},
    {"category": "rag_normal", "question": "¿Qué discotecas o beach clubs hay en Tenerife?"},
    {"category": "rag_normal", "question": "¿Qué puedo comer en la zona norte de Tenerife?"},

    # --- Fuera de alcance: el PDF no debería tener esta información ---
    {"category": "out_of_scope", "question": "¿Qué coches hay disponibles para alquilar?"},
    {"category": "out_of_scope", "question": "¿Cuál es el precio del billete de avión a Tenerife?"},

    # --- Tool-calling: deberían disparar get_city_mobility ---
    {"category": "tool_call_valid", "question": "¿Cómo puedo ir desde la Orotava hasta el Teide?"},
    {"category": "tool_call_valid", "question": "¿Qué transporte me recomiendas para ir de Adeje a Los Cristianos?"},
    {"category": "tool_call_invalid", "question": "¿Cómo puedo ir desde Madrid hasta Tenerife?"},
]

print(f"Total de preguntas en el conjunto de evaluación: {len(evaluation_questions)}")
for item in evaluation_questions:
    print(f"- [{item['category']}] {item['question']}")

Total de preguntas en el conjunto de evaluación: 9
- [rag_normal] ¿Cómo puedo subir al Teide?
- [rag_normal] Dame un listado de playas
- [rag_normal] ¿Qué discotecas o beach clubs hay en Tenerife?
- [rag_normal] ¿Qué puedo comer en la zona norte de Tenerife?
- [out_of_scope] ¿Qué coches hay disponibles para alquilar?
- [out_of_scope] ¿Cuál es el precio del billete de avión a Tenerife?
- [tool_call_valid] ¿Cómo puedo ir desde la Orotava hasta el Teide?
- [tool_call_valid] ¿Qué transporte me recomiendas para ir de Adeje a Los Cristianos?
- [tool_call_invalid] ¿Cómo puedo ir desde Madrid hasta Tenerife?


In [16]:
# EJECUCIÓN DEL CONJUNTO DE EVALUACIÓN
def average_output_tokens(results):
    """Calcula el promedio de tokens de salida consumidos por una lista de turnos.

    Útil para estimar un valor razonable de `max_output_tokens` en base al
    consumo real observado en un conjunto de preguntas de prueba.

    Args:
        results: Lista de diccionarios de turno (los devueltos por
            `run_chat_conversation`), cada uno con la clave `output_tokens`.

    Returns:
        dict: Estadísticas con las claves `avg`, `min`, `max` y `count`.
    """
    token_counts = [r["output_tokens"] for r in results if r.get("output_tokens") is not None]

    if not token_counts:
        return {"avg": 0, "min": 0, "max": 0, "count": 0}

    return {
        "avg": sum(token_counts) / len(token_counts),
        "min": min(token_counts),
        "max": max(token_counts),
        "count": len(token_counts),
    }

def run_evaluation_tool(llm_response):
    """Ejecuta la tool solicitada por el modelo, si la hay.

    Returns:
        tuple: (tool_name o None, tool_result dict o None)
    """
    function_calls = list(llm_response.function_calls or [])
    if not function_calls:
        return None, None

    call = function_calls[0]
    args = dict(call.args)

    if call.name == "get_city_mobility":
        tool_result = get_city_mobility(**args)
    else:
        raise ValueError(f"Herramienta no soportada: {call.name}")

    return call.name, tool_result


def run_evaluation_question(chat, item: dict, k: int = 3) -> dict:
    """Ejecuta una pregunta del conjunto de evaluación y recoge sus métricas.

    Args:
        item: Diccionario con "category" y "question".
        k: Número de documentos a recuperar del vector store.

    Returns:
        dict con question, category, answer, top_similarity_score,
        tool_called, tool_result y output_tokens.
    """
    question = item["question"]

    retrieved_with_scores = vector_store.similarity_search_with_relevance_scores(question, k=k)
    retrieved_docs = [doc for doc, _ in retrieved_with_scores]
    top_similarity_score = retrieved_with_scores[0][1] if retrieved_with_scores else None
    context = format_docs_with_sources(retrieved_docs)

    
    response = chat.send_message(message=f"context: {context}; user question: {question}")

    tool_name, tool_result = run_evaluation_tool(response)

    final_response = response
    if tool_name is not None:
        final_response = chat.send_message(
            genai_types.Part(
                function_response=genai_types.FunctionResponse(
                    name=tool_name,
                    response={"result": tool_result}
                )
            )
        )

    return {
        "category": item["category"],
        "question": question,
        "answer": final_response.text,
        "top_similarity_score": top_similarity_score,
        "tool_called": tool_name,
        "tool_result": tool_result,
        "output_tokens": final_response.usage_metadata.candidates_token_count,
    }


evaluation_results = []
chat = llm_client.chats.create(model=LLM_MODEL, config=model_config)
for item in evaluation_questions:
    result = run_evaluation_question(chat, item, k=3)
    evaluation_results.append(result)

    print(f"[{result['category']}] {result['question']}")
    print(f"  Respuesta: {result['answer']}")
    print(f"  Top similarity score: {result['top_similarity_score']}")
    print(f"  Tool llamada: {result['tool_called']}")
    if result["tool_result"] is not None:
        print(f"  Resultado tool: {json.dumps(result['tool_result'], ensure_ascii=False)}")
    print(f"  Output tokens: {result['output_tokens']}")
    print("-" * 100)

output_tokens_stats = average_output_tokens(evaluation_results)
print("ESTADISTICAS DE TOKENS DE SALIDA")
print(f"Promedio: {output_tokens_stats['avg']:.2f}")
print(f"Minimo:   {output_tokens_stats['min']}")
print(f"Maximo:   {output_tokens_stats['max']}")
print(f"Muestras: {output_tokens_stats['count']}")

[rag_normal] ¿Cómo puedo subir al Teide?
  Respuesta: Para subir al Teide, se recomienda tomar la carretera TF-24 desde la rotonda de Padre Anchieta en La Laguna, haciendo paradas en el Mirador de La Tarta y el Mirador de Chipeque. Una vez en la zona, puedes aparcar en el Parador de las Cañadas del Teide para visitar el mirador de La Ruleta. Para acceder al pico, puedes utilizar el teleférico, y si buscas información, el Centro de Visitantes de El Portillo es gratuito. Además, subir de noche cuando el cielo está despejado permite observar uno de los cielos estrellados más espectaculares del mundo.

TENERIFE.pdf, páginas 14-15
  Top similarity score: 0.6353321075439453
  Tool llamada: None
  Output tokens: 150
----------------------------------------------------------------------------------------------------
[rag_normal] Dame un listado de playas
  Respuesta: Aquí tienes un listado de playas mencionadas en la guía:

*   **Zona Norte:** Playa de Los Patos, Playa del Ancón y Playa de Ben

## LIMITACIONES Y CASOS LÍMITE

- **Retrieval**: el índice se basa en un único PDF y un `k` fijo, sin re-ranking. Preguntas fuera de ese documento (alojamiento, vuelos, alquiler de coches, precios) quedan fuera de alcance por diseño, y los `similarity scores` sirven como referencia relativa, no como umbral absoluto de calidad.

- **Alucinaciones**: en preguntas `out_of_scope` se espera que el modelo admita que no tiene la información. No hay verificación automática de *faithfulness*; la validación de que la respuesta esté soportada por el contexto es manual.

- **Tool-calling**: en las categorías `tool_call_*` puede ocurrir que el modelo no invoque `get_city_mobility` cuando se esperaba, o que la herramienta devuelva un error (ciudad no encontrada, modo inválido) que el modelo debe comunicar sin fallar. Los datos de movilidad son simulados, no proceden de una API real.

- **Tokens y conversación**: al usar un único `chat` para todo el conjunto, el consumo de `input_tokens` crece con el historial acumulado, y las respuestas pueden truncarse si se alcanza `max_output_tokens`.

- **Reproducibilidad**: `LLM_SEED` reduce la variabilidad pero no garantiza determinismo total. El conjunto `evaluation_questions` debe reutilizarse como referencia al comparar cambios de modelo, prompts o parámetros de chunking/retrieval.